# 08 — Evaluation, diagnostics and reporting

**Prepared, not run here:** load saved predictions or run notebook 07 first. Generate diagnostic plots and reporting tables; never claim metrics without execution.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

## Prerequisite

Run notebook `07_machine_learning_models.ipynb` in the same Spark session so `models`, `best_name`, `valid_vec`, `test_vec` and `zone_groups` exist. Alternatively persist their predictions first. This notebook intentionally does not fabricate metrics.

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
chosen=models[best_name].transform(valid_vec).withColumn("prediction",F.greatest(F.lit(0.0),F.col("prediction")))
with_groups=chosen.join(zone_groups.select("PULocationID","demand_category"),"PULocationID")
summary=with_groups.groupBy("demand_category").agg(
 F.count("*").alias("n"),
 F.avg(F.abs(F.col("label")-F.col("prediction"))).alias("MAE_ML"),
 F.avg(F.abs(F.col("label")-F.col("lag_168"))).alias("MAE_weekly_baseline"),
 F.sqrt(F.avg(F.pow(F.col("label")-F.col("prediction"),2))).alias("RMSE_ML"))
summary.orderBy("demand_category").show(truncate=False)

In [ ]:
# Plot actual vs predicted for ONE illustrative zone (midtown center)
plot=(chosen.filter(F.col("PULocationID")==161)
      .orderBy("pickup_hour").select("pickup_hour","label","prediction","lag_168").toPandas())
fig,ax=plt.subplots(figsize=(12,5))
ax.plot(plot.pickup_hour,plot.label,label="Actual")
ax.plot(plot.pickup_hour,plot.prediction,label="Selected ML")
ax.plot(plot.pickup_hour,plot.lag_168,label="Previous week",alpha=.65)
ax.set(title="Midtown Center: Validation Hourly Demand",xlabel="Time",ylabel="Recorded pickups")
ax.legend();ax.grid(alpha=.25);fig.autofmt_xdate();fig.tight_layout();plt.show()

## Reporting guidance

Include MAE and RMSE, training/validation boundaries, group-specific errors, zero-heavy-zone caveats, data completeness assumptions, and the fact that Jan 26–31 was previously inspected. For a genuine final prospective test, acquire a later month and recompute the features without using unseen future information.